# ODI Conversion: HR.TRG_LOC Load

**Conversion Timestamp:** 2024-07-30T12:00:00Z

This notebook loads data into `workspace.hr.trg_loc` from `workspace.hr.locations`.

In [ ]:
# COMMAND ----------
dbutils.widgets.text("ETL_JOB_TYPE", "FULL_LOAD", "1. ETL Job Type (e.g., FULL_LOAD, INCREMENTAL)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "1", "2. Data Source Numeric ID")
dbutils.widgets.text("ETL_PROC_WID", "-1", "3. ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "-1", "4. ODI Session Number")

# ETL Parameters

The following parameters control the execution of this notebook. They are set via Databricks widgets.

In [ ]:
# COMMAND ----------
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  '${DATASOURCE_NUM_ID}' AS datasource_num_id,
  '${ETL_PROC_WID}' AS etl_proc_wid,
  '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
# COMMAND ----------
display(spark.sql("SELECT * FROM v_etl_parameters"))

# Target Load: HR.TRG_LOC

This section performs the main data load into the target table `workspace.hr.trg_loc`.

In [ ]:
# COMMAND ----------
-- SCEN_TASK_NO in {10}
-- SCEN_TASK_NO in {20}
-- SCEN_TASK_NO in {30}
INSERT INTO workspace.hr.trg_loc
  (
    LOCATION_ID ,
    STREET_ADDRESS ,
    POSTAL_CODE ,
    CITY ,
    STATE_PROVINCE ,
    COUNTRY_ID 
  ) 
SELECT 
  LOCATIONS.LOCATION_ID ,
  LOCATIONS.STREET_ADDRESS ,
  LOCATIONS.POSTAL_CODE ,
  LOCATIONS.CITY ,
  LOCATIONS.STATE_PROVINCE ,
  LOCATIONS.COUNTRY_ID  
FROM 
  workspace.hr.locations AS LOCATIONS;

# Validation

Verify the loaded records in the target table.

In [ ]:
# COMMAND ----------
%sql
SELECT COUNT(*) AS total_records_in_trg_loc
FROM workspace.hr.trg_loc;

In [ ]:
# COMMAND ----------
%sql
SELECT *
FROM workspace.hr.trg_loc
LIMIT 10;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** All Oracle schema references (`HR`) have been converted to `workspace.hr` and table names to lowercase (`trg_loc`, `locations`) as per Databricks naming conventions.
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is specific to Oracle and not applicable to Delta Lake.
3.  **ODI SCEN_TASK_NO:** The `SCEN_TASK_NO` markers (`{10}`, `{20}`, `{30}`) were comments in the source and are preserved as comments in the relevant SQL cell.
4.  **Error Handling & Logging:** Original ODI error handling and logging mechanisms are not directly converted. It is assumed that Databricks job monitoring and Spark's native error handling will manage this. If specific error tables (`E$_`) or check tables (`SNP_CHECK_TAB`) were present, they would be handled as persistent Delta tables with appropriate DML.
5.  **Deduplication/Updates:** This specific ODI script only contained an `INSERT`. If future scripts contain `UPDATE`, `DELETE`, or `MERGE` logic, they will be converted following the rules outlined in Section 7 of the System Prompt, particularly handling the `IND_UPDATE` pattern with Databricks `MERGE INTO`.
6.  **Data Type Conversion:** As the source only provided an `INSERT ... SELECT`, no explicit `CREATE TABLE` DDL was present. If it were, Oracle data types would be mapped to Spark SQL types as per Section 5 of the System Prompt (e.g., `VARCHAR2` to `STRING`, `NUMBER(p,0)` to `BIGINT`, `DATE` to `TIMESTAMP`).
7.  **Widgets:** Placeholder widgets for `ETL_JOB_TYPE`, `DATASOURCE_NUM_ID`, `ETL_PROC_WID`, and `ODI_SESS_NO` have been included as per the standard notebook structure, although not explicitly used in this simple `INSERT` statement.